# Publication text pass

Runs the text pass from `src/helper.py` on the papers in `data/publications/Publications`.
Each paper gets `outputs/publication_structures/<paper>/text_pass/` with:

- `structure.json` — the paper as a tree (front matter, sections, paragraphs, references)
- `observations.json` — every line in reading order; the input for the OpenAI/Claude notebook
- `audit.json` — what was and wasn't identified

The vision stage (GPU) is not used.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, logging
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
logging.getLogger("pdfminer").setLevel(logging.ERROR)

from src.helper import (
    configure, show_table, publish_json, flat_text, check_flat_text,
    read, walk, run_text_pass, run_all, SCHEMA_PATH,
)

## 1. Choose a paper

In [ ]:
config = configure("2023_QM034_Synthesis_of_ultra_pure_Co3Sn2S2_crystals.pdf")
print("PDF:", config["pdf"].name)
print("Output folder:", config["output"])

## 2. Flat page text (baseline)

In [ ]:
flat_document = flat_text(config)
check_flat_text(flat_document)
print("Pages:", len(flat_document["pages"]))
print("Text characters:", sum(len(p["text"]) for p in flat_document["pages"]))

In [ ]:
publish_json(
    config["output"] / "json_stages" / "01_flat_page_text.json",
    flat_document,
    "Stage 1 — Complete page-text baseline",
)

## 3. Structured text pass

In [ ]:
result = run_text_pass(config["pdf"].name, force=True)
result

In [ ]:
out = config["output"] / "text_pass"
tree = read(out / "structure.json")
audit = read(out / "audit.json")
print("Checks:", audit["validation"])
print("Every line placed in the tree:", audit["coverage_complete"])
show_table(
    [{"type": t, "count": c} for t, c in Counter(n["type"] for n in walk(tree["document"])).most_common()],
    ["type", "count"],
)

### Sections found

In [ ]:
show_table(
    [
        {"type": n["type"], "title": n.get("title", ""), "pages": f'{n["page_start"]}–{n["page_end"]}'}
        for n in walk(tree["document"])
        if n["type"] in ("abstract", "section", "subsection", "acknowledgements", "data_availability", "references")
    ],
    ["type", "title", "pages"],
)

### The file the LLM notebook reads

In [ ]:
observations = read(out / "observations.json")
print(len(observations), "lines")
show_table(observations[:15], ["id", "page", "reading_order", "text"])

## 4. All papers

Skips papers that already have `observations.json`; use `run_all(force=True)` to redo them.
A failure on one paper is recorded and the loop continues.

In [ ]:
results = run_all()
show_table(results, ["paper", "status", "lines", "unidentified", "error"])